# tensor-item-scalar — ex7: early stopping via .item() threshold

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-item-scalar`. Running the final beacon cell reports progress against the `Numpy: Core array literacy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-item-scalar`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-item-scalar"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch `.item()` — quick refresher

`tensor.item()` extracts a Python scalar (`int`, `float`, or `bool`) from a 0-D or 1-element tensor. It detaches from the autograd graph and pulls the value back to CPU. This is the canonical bridge from tensor-world to Python-world: logging, control flow, plotting, and stop conditions all need scalars.

**Calling `.item()` on a multi-element tensor raises.** Use `.tolist()` if you want every element as a Python list. Calling `.item()` inside a hot inner loop forces a CPU sync — fine for diagnostics, expensive in the training step itself.

### Exercise 7 — early stopping via .item() threshold

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Use `.item()` to extract a scalar criterion each iteration and drive a Python `while` loop that stops when the criterion falls below a threshold.
> Keywords: early-stopping, control-flow, threshold, while-loop
> ```

**KCs targeted:** `item-zero-d-extract`, `item-control-flow`

Implement `ex7_iterate_until(x_init, decay, threshold, max_iter)`. The canonical use of `.item()` in control flow:

1. Start from `x = x_init.clone()`.
2. For up to `max_iter` iterations:
   a. Compute the L2 norm: `crit = x.norm()` (a 0-D tensor).
   b. Extract the scalar with `crit.item()`. **Print** `'iter {i}: norm={value:.4f}'` so you can see the descent.
   c. If `crit.item() < threshold`, break out of the loop.
   d. Otherwise, decay: `x = x * decay`.
3. Return a dict: `{'x': x, 'iters_done': i + 1, 'final_norm': crit.item(), 'stopped_early': bool}`.

Why `.item()` and not just `if crit < threshold`? In a `while` condition or `if` statement, Python implicitly calls `bool()` on the tensor. For a 0-D tensor this works but emits warnings in some torch versions; for multi-element tensors it raises. Calling `.item()` explicitly is the unambiguous, fast, and future-proof move.

Inputs:
- `x_init`: 1-D float tensor.
- `decay`: float in (0, 1).
- `threshold`: float > 0.
- `max_iter`: int.

The visualization plots `norm` vs iteration and marks the early-stop point if it triggered.

In [ ]:
def ex7_iterate_until(x_init: Tensor, decay: float, threshold: float, max_iter: int) -> dict:
    x = x_init.clone()
    stopped_early = False
    final_norm = float('inf')
    i = 0
    for i in range(max_iter):
        crit = x.norm()
        val = crit.item()
        print(f'  iter {i}: norm={val:.4f}')
        final_norm = val
        if val < threshold:
            stopped_early = True
            break
        x = x * decay
    return {
        'x': x,
        'iters_done': i + 1,
        'final_norm': final_norm,
        'stopped_early': stopped_early,
    }


<details><summary>Solution</summary>

```python
def ex7_iterate_until(x_init: Tensor, decay: float, threshold: float, max_iter: int) -> dict:
    x = x_init.clone()
    stopped_early = False
    final_norm = float('inf')
    i = 0
    for i in range(max_iter):
        crit = x.norm()
        val = crit.item()
        print(f'  iter {i}: norm={val:.4f}')
        final_norm = val
        if val < threshold:
            stopped_early = True
            break
        x = x * decay
    return {
        'x': x,
        'iters_done': i + 1,
        'final_norm': final_norm,
        'stopped_early': stopped_early,
    }
```

**`.item()` is the explicit form of tensor → Python.** PyTorch will sometimes implicitly convert (e.g. `if x:` on a 0-D bool tensor) but the rules are quietly version-dependent. Calling `.item()` is unambiguous, fast (single CPU read), and works the same in every torch release.

**CPU sync cost.** On GPU, `.item()` forces a host-device sync — the CPU has to wait for the GPU to finish the op that produced the scalar. In a hot training loop, calling `.item()` every step can dominate runtime. Reserve it for periodic logging and stop conditions, not for things you do per-batch.

**Sentinel `iters_done`.** Returning `i + 1` works whether you broke early or completed all iterations because `i` survives after the `for` loop ends in Python (it doesn't have its own scope).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()